# SAM zero-shot trên Colab (VS Code hoặc web)

Yêu cầu:
- Kernel là Colab có **GPU** (T4 miễn phí là đủ).
- Google Drive có `MyDrive/data_segmentation/data/` với cấu trúc giống `data/` của repo (`raw/`, `processed/`, `splits/`, `metadata/`).

Kết quả (`outputs/sam/`) và log (`logs/`) được ghi thẳng vào `MyDrive/data_segmentation/results/`, nên runtime bị ngắt thì chạy lại các ô là tiếp tục từ chỗ dừng.

In [ ]:
REPO_URL = "https://github.com/quanai06/image_segmentation.git"
BRANCH = "quan"
DRIVE_ROOT = "/content/drive/MyDrive/data_segmentation"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results"
REPO = "/content/image_segmentation"

MODEL = "vit_h"          # vit_b | vit_l | vit_h
SPLIT = "test"           # test | all
PROTOCOLS = ""           # "" = tất cả P0–P4; hoặc vd "box points"

## 1. Nối Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
assert os.path.isdir(DRIVE_DATA), f"Không thấy {DRIVE_DATA}"
!ls {DRIVE_DATA}

## 2. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có GPU")

## 3. Lấy code nhánh `quan`

In [ ]:
%cd /content
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -1

## 4. Cài thư viện còn thiếu

Không cài theo `requirements.txt` để tránh xung đột với các gói Colab cài sẵn.

In [ ]:
!pip install -q segment-anything albumentations==2.0.8 tifffile
!pip install -q -e . --no-deps

## 5. Chép dữ liệu từ Drive vào máy ảo

Đọc trực tiếp từ Drive rất chậm, nên chép `raw/` và `processed/` vào máy ảo một lần. `splits/` và `metadata/` lấy từ Git (bản chuẩn); ô dưới kiểm tra bản trên Drive có khớp không.

In [ ]:
for sub in ["raw", "processed"]:
    !rsync -a --exclude archives {DRIVE_DATA}/{sub}/ data/{sub}/
!du -sh data/raw data/processed
!diff -rq {DRIVE_DATA}/splits data/splits && echo "splits trên Drive khớp với Git"
!python -m pytest -q tests/test_data_pipeline.py

## 6. Ghi kết quả và log vào Drive

In [ ]:
os.makedirs(f"{DRIVE_RESULTS}/outputs/sam", exist_ok=True)
os.makedirs(f"{DRIVE_RESULTS}/logs", exist_ok=True)
!rm -rf outputs/sam logs && ln -s {DRIVE_RESULTS}/outputs/sam outputs/sam && ln -s {DRIVE_RESULTS}/logs logs
!ls -la outputs logs

## 7. Tải trọng số SAM

In [ ]:
!python scripts/download_sam_weights.py --models {MODEL}

## 8. Chạy thử 3 ảnh (không ghi kết quả)

In [ ]:
!python scripts/run_sam_zeroshot.py --model {MODEL} --split {SPLIT} --limit 3 --protocols box points --dry-run

## 9. Chạy đầy đủ

Bị ngắt giữa chừng thì chạy lại ô này: script bỏ qua các ảnh đã có trong CSV.

In [ ]:
protocols = f"--protocols {PROTOCOLS}" if PROTOCOLS else ""
!python scripts/run_sam_zeroshot.py --model {MODEL} --split {SPLIT} --save-masks {protocols}

## 10. Tổng hợp và lưu báo cáo lên Drive

In [ ]:
!python scripts/summarize_sam.py --examples-model {MODEL} --examples-split {SPLIT}
!mkdir -p {DRIVE_RESULTS}/report && cp -r report/tables report/figures {DRIVE_RESULTS}/report/
!ls {DRIVE_RESULTS}/report/tables